# 5일차 RAG 미니 프로젝트 · 설비지원센터 문서 RAG 만들기

가상 반도체 공장 설비지원센터 문서 100건으로 사내 지식 Q&A(RAG)를 처음부터 만듭니다. 재료와 평가셋(20문항)은 받아서 쓰고, 평가셋으로 세 번 이상 고친 뒤 마지막에 테스트셋 10문항으로 고친 효과가 처음 보는 질문에서도 유지되는지 확인합니다.

## 진행 순서

- **1단계 · RAG 만들기** 재료 100건과 평가셋을 받고, 명세를 보고 RAG 네 칸(청킹 · 색인 · 검색 · 답변)을 직접 짠 뒤, 평가 도구로 검색 방식 네 가지(임베딩 · 키워드 · 하이브리드 · 리랭커)와 검색기 채점 출력을 봅니다.
- **2단계 · 평가하고 고치기** 틀린 문항을 보고 세 번 이상 고친 뒤, 테스트셋을 한 번 돌리고 개선내역을 써서 제출합니다.
- **3단계 · *내 업무로* 재료와 평가셋 만들기(선택)** 제출을 마친 뒤 시간이 남으면, 재료를 내 업무 문서로 바꿔 같은 노트북을 다시 돌려 봅니다.

## 지킬 것

- **테스트용은 개선과정에선 안쓰고, 마지막에 한 번만 사용합니다.**
- **평가 도구와 채점 프롬프트는 고치지 않습니다.** 모두가 같은 잣대로 재야 개선 전후를 비교할 수 있습니다.
- **이름 약속을 지킵니다.** 평가 도구가 `search_vector` · `search_bm25` · `search_hybrid` · `search_rerank` · `ask` · `Chunk` · `CONFIG` · `REFUSE` 를 부릅니다(1단계 명세 참고).

## 제출물

- 실행 출력이 남은 노트북(.ipynb, 「8. 개선내역」 셀까지 채운 것) · 노트북이 만든 결과 파일(`results.csv` · `fail_*.csv`). 내려받는 법은 「9. 제출」에 있습니다.

## 시작 전 · 파일 이름과 위치 한눈에 보기

이 노트북은 위에서부터 차례로 실행합니다. 「완성 코드」 셀은 그대로 실행하고, 「작성」 셀만 채웁니다.

**① 이 노트북** [파일] → [드라이브에 사본 저장] 으로 내 드라이브에 사본을 만들고, 사본 이름을 `aaa_day05-mini-project.ipynb` 로 바꿉니다. aaa 자리에는 내 식별자를 넣습니다(왼쪽 위 제목을 눌러 바꿉니다). 모두 같은 이름으로 내면 제출물을 구분할 수 없습니다.

**② 재료** 「0. 재료 받기」 셀이 학습 사이트에서 `day05-mini-project-docs.zip` 을 받아 `mini_docs/` 폴더에 풉니다. 문서 한 건이 md 파일 하나입니다. 셀이 zip 을 받지 못하면 학습 사이트 5일차 페이지에서 zip 을 내려받아 왼쪽 📁 파일 창 맨 위 칸(`sample_data` 폴더 바깥)에 올린 뒤 그 셀을 다시 실행합니다.

**③ 평가셋 · 테스트셋** 파일이 아니라 「2. 평가셋 · 테스트셋」 셀 안에 들어 있습니다. 셀에서 바로 읽고 고칠 수 있습니다.

**④ 노트북이 만드는 파일** 모두 파일 창 맨 위 칸에 생깁니다. 지우거나 이름을 바꾸지 않습니다.
- `my_manifest.csv` 문서 목록 · `llm_cache.json` Claude 답 저장(다시 돌려도 요금이 안 나감)
- `results.csv` 평가할 때마다 한 줄씩 쌓이는 점수표 · `fail_<태그>.csv` 문항별 결과

**⑤ 런타임이 끊기면** 오래 쉬거나 [런타임 연결 해제]를 하면 파일 창의 파일이 **모두 지워집니다.** 노트북에 쓴 코드와 평가셋은 남습니다.
- 개선을 한 번 마칠 때마다 `results.csv` 를 파일 창에서 [다운로드] 해 둡니다.
- 끊겼으면 **맨 위 셀부터** 다시 실행합니다(개선 셀까지 다시 돌려야 `results.csv` 가 다시 쌓입니다).

# 1단계 · RAG 만들기

## 0. 재료 받기와 환경 준비 (완성 코드)

세 셀을 차례로 실행합니다. 첫 셀이 재료를 받고 라이브러리를 설치합니다. 코랩 왼쪽 열쇠 아이콘(보안 비밀)에 `ANTHROPIC_API_KEY` 를 넣고 「노트북 액세스」를 켭니다. 2일차에 받은 키입니다. 둘째 셀에 `OK` 와 모델 이름이, 셋째 셀에 「준비 완료」가 찍히면 됩니다.

In [1]:
# @title 0. 재료 받기와 설치 (완성 코드)
# 0. 재료 zip 을 학습 사이트에서 받아 mini_docs/ 폴더에 풀고, 라이브러리를 설치합니다
import os, shutil, zipfile, urllib.request
SITE = 'https://lecture-web-five.vercel.app/materials/'
DOCS_ZIP = 'day05-mini-project-docs.zip'     # 재료 zip 이름 (3단계에서 내 재료로 바꿀 때만 고칩니다)
DOC_DIR = 'mini_docs'                          # 재료를 푸는 폴더
if not os.path.exists(DOCS_ZIP):               # 파일 창에 zip 이 없으면 학습 사이트에서 받습니다
    try:
        urllib.request.urlretrieve(SITE + DOCS_ZIP, DOCS_ZIP)
    except Exception as error:
        print('[안내]', DOCS_ZIP, '를 받지 못했습니다 ·', type(error).__name__,
              '· 학습 사이트 5일차에서 받아 왼쪽 파일 창 맨 위에 올린 뒤 이 셀을 다시 실행합니다')
if os.path.exists(DOCS_ZIP):
    shutil.rmtree(DOC_DIR, ignore_errors=True)  # 전에 푼 재료를 지우고 새로 풉니다
    zipfile.ZipFile(DOCS_ZIP).extractall(DOC_DIR)
    print('재료', DOCS_ZIP, '→', DOC_DIR, '· md 파일', sum(f.endswith('.md') for _, _, fs in os.walk(DOC_DIR) for f in fs), '개')
%pip install -q -U anthropic sentence-transformers rank-bm25 pandas numpy

재료 day05-mini-project-docs.zip → mini_docs · md 파일 100 개
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.


In [3]:
# @title 0-1. 키 넣기와 연결 확인 (완성 코드)
# 0-1. 라이브러리를 불러오고 Claude API 키를 넣은 뒤, 짧은 호출로 연결을 확인합니다
import os, re, json, time, glob, hashlib, threading, getpass
import numpy as np, pandas as pd
from collections import namedtuple, defaultdict
from concurrent.futures import ThreadPoolExecutor
# 코랩이면 보안 비밀(왼쪽 열쇠 아이콘)에 넣은 ANTHROPIC_API_KEY 를 읽습니다
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
except Exception:
    pass
# 보안 비밀이 없으면 키를 직접 입력받습니다(입력한 글자는 화면에 보이지 않습니다)
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Claude API 키: ')

import anthropic
# 일시 오류면 세 번까지 다시 시도하고, 호출 한 번을 최대 300초 기다립니다
client = anthropic.Anthropic(max_retries=3, timeout=300)
# 연결 확인: Haiku 모델에 OK 두 글자만 요청합니다
r = client.messages.create(model='claude-haiku-4-5', max_tokens=20,
                           messages=[{'role': 'user', 'content': '다른 말 없이 OK 두 글자만 답하세요'}])
print(r.content[0].text, '·', r.model)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


OK · claude-haiku-4-5-20251001


In [4]:
# @title 0-2. Claude 호출 함수 준비 (완성 코드)
# 0-2. Claude 호출 함수 llm() 과 답 캐시, 공통 이름(REFUSE · Chunk)을 준비합니다
# 채점을 여러 개 동시에 실행할 때 캐시 파일을 한 번에 하나씩만 쓰도록 잠급니다
LOCK = threading.Lock()
# 전에 받은 답(llm_cache.json)이 있으면 불러옵니다
CACHE = json.load(open('llm_cache.json', encoding='utf-8')) if os.path.exists('llm_cache.json') else {}

def llm(prompt, model='claude-sonnet-5-5', max_tokens=1500):
    """Claude 호출 한 번. 같은 (모델, 프롬프트)는 llm_cache.json 에서 꺼내므로 다시 실행해도 요금이 나가지 않습니다."""
    # (모델, 프롬프트)로 16자리 지문을 만들어 캐시 열쇠로 씁니다
    h = hashlib.sha256(json.dumps([model, prompt], ensure_ascii=False).encode()).hexdigest()[:16]
    if h in CACHE:
        return CACHE[h]
    kw = {} if 'haiku' in model else {'output_config': {'effort': 'low'}}   # Haiku 는 effort 를 받지 않습니다
    r = client.messages.create(model=model, max_tokens=max_tokens,
                               messages=[{'role': 'user', 'content': prompt}], **kw)
    out = ''.join(b.text for b in r.content if b.type == 'text').strip()      # thinking 블록은 건너뜁니다
    # 답이 max_tokens 에서 끊겼으면 끝에 표시를 붙입니다
    if r.stop_reason == 'max_tokens':
        out += ' [잘림]'
    with LOCK:
        CACHE[h] = out
        json.dump(CACHE, open('llm_cache.json', 'w', encoding='utf-8'), ensure_ascii=False)
    return out

REFUSE = '문서에서 근거를 찾지 못했습니다'          # 문서에 없는 질문의 답에는 이 문구가 들어가야 합니다
Chunk = namedtuple('Chunk', 'doc_id version date title section text')

print('준비 완료 · 캐시', len(CACHE), '건')

준비 완료 · 캐시 0 건


## 1. 재료 읽기 (완성 코드)

`mini_docs/` 의 md 파일을 한 건씩 읽어 문서 100건을 만듭니다. 출력 표에서 문서 수 · 길이 · 개정판을 확인합니다.

- 문서는 노트북 메모리의 파이썬 목록 `DOCS` 에 한 건씩 딕셔너리로 담깁니다. md 파일 맨 위의 제목 줄 · 문서 정보 네 줄과 본문이 아래처럼 들어갑니다.

```python
DOCS[0] = {
    'key':     'ET-PRT-301_v1',              # 문서번호_판 (중복 검사용)
    'doc_id':  'ET-PRT-301',                 # 「- 문서번호:」 줄 · 평가셋의 정답 문서 번호와 맞추는 값
    'version': 'v1',                         # 「- 판:」 줄 · 개정판은 v2
    'date':    '2025-03-14',                 # 「- 시행일:」 줄
    'kind':    '부품 교체 기준표',            # 「- 종류:」 줄
    'title':   'EC-4 식각 장비 소모품 교체 기준표',   # 맨 위 「# 」 줄의 제목
    'text':    '## 1. 적용 범위\n이 문서는 …',      # 본문(md, ## 절 제목 포함)
}
```
- 3일차 예제와 견주면 `names`(파일 이름 목록) → `d['doc_id']`, `docs`(본문 목록) → `d['text']`, `corpus_manifest.csv` 의 꼬리표 → `d['version']` · `d['date']` · `d['kind']` 입니다. 칸 A 는 파일을 열지 않고 이 `DOCS` 를 읽습니다.
- 본문을 뺀 목록표만 `my_manifest.csv` 로 저장합니다. 런타임이 끊기면 `DOCS` 도 사라지므로, 맨 위 셀부터 다시 실행합니다.
- 「md 파일이 없습니다」로 멈추면 「0. 재료 받기」 셀을 다시 실행합니다.

In [5]:
# @title 1. 재료 읽기 (완성 코드)
# 1. mini_docs/ 의 md 파일을 한 건씩 읽습니다. 파일 맨 위 「# 제목」 줄과 「- 문서번호: …」 같은 정보 줄을 꺼내고 나머지를 본문으로 둡니다
import os, re, glob
paths = sorted(glob.glob(os.path.join(DOC_DIR, '**', '*.md'), recursive=True))
assert paths, f'{DOC_DIR}/ 에 md 파일이 없습니다 · 「0. 재료 받기」 셀을 다시 실행합니다'
FIELDS = {'문서번호': 'doc_id', '판': 'version', '시행일': 'date', '종류': 'kind'}
DOCS, seen = [], set()
for p in paths:
    raw = open(p, 'rb').read()
    try:
        text = raw.decode('utf-8-sig')
    except UnicodeDecodeError:                                   # 메모장에서 ANSI 로 저장한 경우
        text = raw.decode('cp949')
    lines = text.replace('\r\n', '\n').split('\n')
    stem = os.path.splitext(os.path.basename(p))[0]              # 정보 줄이 없으면 파일 이름에서 꺼냅니다(예: MY-GDE-001_v2)
    d = {'doc_id': stem.split('_')[0], 'version': (stem.split('_')[1] if '_' in stem else 'v1').lower(), 'date': '', 'kind': '', 'title': stem}
    body_start = 0
    for n, ln in enumerate(lines[:12]):                          # 정보 줄은 파일 맨 위 12줄 안에서만 찾습니다
        m = re.match(r'^\s*-\s*(문서번호|판|시행일|종류)\s*:\s*(.+?)\s*$', ln)
        if ln.startswith('# ') and n < 3:
            d['title'] = ln[2:].strip(); body_start = n + 1
        elif m:
            d[FIELDS[m.group(1)]] = m.group(2).lower() if m.group(1) == '판' else m.group(2); body_start = n + 1
    if d['title'].startswith(d['doc_id']):                       # 제목 앞의 문서번호는 뗍니다
        d['title'] = d['title'][len(d['doc_id']):].strip()
    d['text'] = '\n'.join(lines[body_start:]).strip()
    d['key'] = f"{d['doc_id']}_{d['version']}"
    assert d['key'] not in seen, f"같은 문서번호 · 판이 두 번 나옵니다: {d['key']} ({p})"
    assert d['text'], f'본문이 비어 있습니다: {p}'
    seen.add(d['key']); DOCS.append(d)

M = pd.DataFrame([{k: d[k] for k in ('doc_id', 'version', 'date', 'kind', 'title')} | {'글자 수': len(d['text'])} for d in DOCS])
print(M.head(8).to_string(index=False)); print('…')
print(f"\n문서 {len(DOCS)}건 · 평균 {int(M['글자 수'].mean()):,}자 · 가장 짧은 문서 {int(M['글자 수'].min()):,}자 · 개정판 {(M.version != 'v1').sum()}건")
print('종류별:', M.kind.value_counts().to_dict())
M.to_csv('my_manifest.csv', index=False, encoding='utf-8-sig')

     doc_id version       date      kind                                      title  글자 수
 CL-ALM-101      v1 2025-05-12 알람 조치 가이드     WS-5 세정 장비 약액 농도 이탈 알람(코드 W858) 조치 가이드  2594
 CL-ALM-102      v1 2025-06-02 알람 조치 가이드    WS-5 세정 장비 린스 비저항 저하 알람(코드 W352) 조치 가이드  2420
 CL-ALM-103      v1 2024-09-24 알람 조치 가이드 WS-5 세정 장비 건조부 질소 압력 저하 알람(코드 W441) 조치 가이드  2515
 CL-ALM-104      v1 2024-03-01 알람 조치 가이드     WS-5 세정 장비 배스 온도 이탈 알람(코드 W847) 조치 가이드  2422
 CL-ALM-105      v1 2024-07-25 알람 조치 가이드   WS-5 세정 장비 파티클 카운트 초과 알람(코드 W243) 조치 가이드  2467
 CL-ALM-106      v1 2024-11-07 알람 조치 가이드   WS-5 세정 장비 로봇 웨이퍼 미검출 알람(코드 W116) 조치 가이드  2644
CL-CASE-707      v1 2025-03-11 장애 사례 보고서               WS-5 세정 로봇 웨이퍼 미검출 장애 사례 보고서  2448
CL-CASE-708      v1 2025-09-12 장애 사례 보고서             WS-5 세정 건조부 질소 압력 저하 장애 사례 보고서  2779
…

문서 100건 · 평균 2,643자 · 가장 짧은 문서 2,342자 · 개정판 6건
종류별: {'알람 조치 가이드': 32, '정기 점검 절차서': 16, '장애 사례 보고서': 15, '운영 규정': 13, '부품 교체 기준표': 11, '변경 승인 절차': 7, '안전 작업 기준': 6}


## 2. 평가셋 · 테스트셋 (제공 · 고칠 수 있음)

아래 셀에 평가셋 20문항 · 테스트셋 10문항이 들어 있습니다. 셀에서 바로 읽고, 필요하면 고친 뒤 검사 셀을 다시 실행합니다. 검사 셀이 정답 문서 번호와 유형을 확인합니다.

**한 문항은 네 칸입니다.**
- **질문** 동료에게 실제로 물어볼 말투로 썼습니다.
- **정답 문서 번호** 문서의 「- 문서번호:」 값입니다(`ET-ALM-102`). 두 문서를 봐야 하면 쉼표로 잇고(`CM-CASE-708, FC-OPS-603`), 문서에 없는 질문이면 `없음` 입니다. 판(v1 · v2)은 쓰지 않습니다.
- **정답 요지** 답의 핵심 값을 한 줄로 씁니다. 채점 모델이 이 줄과 정답 문서 원문을 보고 맞았는지 판정합니다. 판본 질문은 개정판의 값입니다.
- **유형** 식별자(문서 번호 · 코드로 묻기) · 바꿔말하기(문서에 없는 말로 묻기) · 판본(개정판에서 바뀐 값을 묻기) · 여러문서(두 문서를 같이 봐야 답) · 없음(문서에 없는 질문) · 자유

In [6]:
# 2. 평가셋 · 테스트셋 (제공 · 고칠 수 있음)
# (질문, 정답 문서 번호, 정답 요지, 유형)
# 정답 문서 번호: 'ET-ALM-102' · 'CM-CASE-708, FC-OPS-603' · '없음'
# 유형: 식별자 · 바꿔말하기 · 판본 · 여러문서 · 없음 · 자유
EVALSET = [
    ('CP-3 C912 알람은 종점 신호가 설정 연마 시간의 몇 %까지 안 잡히면 뜨는 거야?', 'CM-ALM-104', '설정 연마 시간의 130%', '식별자'),
    ('IM-1 I739 알람 나면 설비 상황실에 몇 분 안에 1차 보고해야 돼?', 'IM-ALM-103', '알람 발생 후 20분 이내', '식별자'),
    ('FC-OPS-603에서 부품 교체 이력 올릴 때 MAT-ON 출고 번호 몇 자리 넣으라고 했지?', 'FC-OPS-603', '12자리', '식별자'),
    ('WS-5 W243 알람 뜨면 알람 시각이랑 측정값은 몇 분 안에 EQ-LOG에 적어야 돼?', 'CL-ALM-105', '알람 발생 후 10분 안', '식별자'),
    ('증착 장비 샤워헤드 온도 알람 났을 때, 식히는 물 흐르는 양은 최소 얼마는 나와야 정상이에요?', 'DP-ALM-101', '냉각수 유량 3.2 L/min 이상', '바꿔말하기'),
    ('식각 장비 주간 점검할 때 정전척 뒷면 헬륨 새는 양은 얼마까지 괜찮아요?', 'ET-PM-201', '1.5 sccm 이하', '바꿔말하기'),
    ('연마 장비 세정부에 새 브러시 끼운 뒤 길들일 때 초순수 유량은 얼마로 맞춰요?', 'CM-ALM-106', '분당 1.2 L (10분간)', '바꿔말하기'),
    ('식각 장비 부품 갈고 나서 챔버 내벽 길들이기용 웨이퍼는 몇 장 흘려요?', 'ET-PRT-301', '시즈닝 웨이퍼 25장', '바꿔말하기'),
    ('식각 장비에 쓸 예비 부품을 밤에 급히 내주려면, 당직 승인 나고 몇 시간 안에 나와?', 'FC-OPS-610', '당직 승인 후 4시간 이내', '바꿔말하기'),
    ('EC-4 챔버 압력 상승 알람 나면 설비지원센터 당직한테 몇 분 안에 보고해야 해?', 'ET-ALM-101', '10분 이내 (v2 개정값)', '판본'),
    ('증착 장비 DP-2 월간 PM은 작업 시간을 최대 몇 시간까지 잡아?', 'DP-PM-202', '최대 8시간 (v2 개정값)', '판본'),
    ('PM 주기를 긴급 변경으로 바꿨는데 정규 절차를 못 마치면 며칠 지나서 이전 주기로 되돌려?', 'FC-CHG-402', '효력 14일, 그 안에 못 마치면 이전 주기로 (v2 개정값)', '판본'),
    ('EC-4 식각 장비 RF 반사 전력 초과로 공정 멈췄던 2025-10-27 건, 총 정지 시간이 얼마였어? 그리고 같은 알람 조치 가이드 기준으로는 복구에 몇 시간 이상 걸리면 설비지원센터장에게 보고해야 해?', 'ET-CASE-711, ET-ALM-102', '총 정지 11시간 20분 / 복구 2시간 이상이면 센터장 보고', '여러문서'),
    ('CP-3 CMP 장비 연마 패드 말인데, 주간 PM 점검표의 잔여 두께 기준값이랑 소모품 교체 기준표에서 조기 교체 대상으로 분류하는 잔여 두께는 각각 몇 mm야?', 'CM-PM-201, CM-PRT-301', '주간 PM 기준 0.8 mm 이상 / 조기 교체 0.9 mm 이하', '여러문서'),
    ('IM-1 주간 PM 시작하려면 마지막 빔 가동 끝나고 최소 몇 분 지나 있어야 해? 그리고 그 PM에서 LOTO 걸 때 잔류 에너지 방출 대기는 몇 분이야?', 'IM-PM-201, FC-SAF-501', '빔 정지 후 30분 / 잔류 에너지 방출 대기 5분', '여러문서'),
    ('EC-4 식각 장비 RF 발생기 한 대 구매 단가가 얼마야?', '없음', '문서에 없음', '없음'),
    ('WS-5 세정 장비 담당 협력사 엔지니어 휴대폰 번호 알려줘', '없음', '문서에 없음', '없음'),
    ('설비지원센터 야간 당직자 식대는 한 끼에 얼마 지원돼?', '없음', '문서에 없음', '없음'),
    ('WS-5 질소 압력 알람 조치 끝내고 EQ-LOG 기록은 언제까지 남겨야 해요?', 'CL-ALM-103', '조치를 마치고 24시간 안', '자유'),
    ('EQ-LOG가 30분 넘게 먹통이라 종이 양식에 적어뒀는데, 복구되면 몇 시간 안에 시스템에 다시 입력해야 돼?', 'FC-OPS-603', '복구 후 4시간 이내', '자유'),
]
TESTSET = [
    ('EC-4 월간 PM은 작업 시간을 몇 시간 이내로 잡아?', 'ET-PM-202', '5시간 이내', '식별자'),
    ('IM-CASE-703 건(IM-1 알람 BC-4417) 때 필라멘트 사용 시간이 몇 시간이었어?', 'IM-CASE-703', '412시간', '식별자'),
    ('이온주입 장비 월간 점검에서 척 뒷면 헬륨 새는 양은 얼마까지 괜찮아요?', 'IM-PM-202', '0.8 sccm 이하', '바꿔말하기'),
    ('세정 장비 월간 점검하다가 케미칼 빼내거나 갈아야 하면 몇 도 밑으로 식은 다음에 시작해?', 'CL-PM-202', '40 ℃ 이하', '바꿔말하기'),
    ('식각 장비에서 식각 끝점 못 잡는 알람 나서 관찰 창 닦았는데, 닦고도 빛 통과율이 몇 % 미만이면 창 갈아야 해?', 'ET-ALM-104', '75% 미만', '바꿔말하기'),
    ('CP-3 헤드 진공 저하 알람 조치하고 재가동하면 몇 시간 동안 재발 여부를 지켜봐야 해?', 'CM-ALM-103', '재가동 후 8시간 (v2 개정값)', '판본'),
    ('CP-3 CMP 슬러리 유량 저하(SLR-F-214) 재발 방지로 필터 교체 주기를 몇 시간으로 줄이기로 했어? 그 변경 관리 문서 기준으로 센터장은 파트장 승인 후 얼마 안에 처리해야 해?', 'CM-CASE-704, FC-CHG-403', '누적 300시간 / 1업무일 이내', '여러문서'),
    ('EC-4 배기 밸브 응답 지연 알람(E150) 때문에 구동부 교체하려는데, 질소 퍼지는 몇 분 하고 챔버 냉각은 몇 분 이상 해야 돼?', 'ET-ALM-105, ET-PRT-302', '질소 퍼지 10분 / 챔버 냉각 60분 이상', '여러문서'),
    ('IM-1 이온주입 장비는 몇 년도에 들여왔어?', '없음', '문서에 없음', '없음'),
    ('CP-3 연마 패드는 어느 업체에서 납품받아?', '없음', '문서에 없음', '없음'),
]

In [7]:
# @title 2. 평가셋 · 테스트셋 검사 (완성 코드)
# 2. 평가셋 · 테스트셋 형식을 검사해 파일로 저장합니다
IDS = {d['doc_id'] for d in DOCS}

# 문항마다 네 칸 · 빈 칸 · 유형 · 정답 문서 번호를 검사하고 id(E01 · T01 …)를 붙여 돌려줍니다
def check_set(S, name, pre):
    out = []
    for i, row in enumerate(S, 1):
        assert len(row) == 4, f'{name} {i}번은 (질문, 정답 문서 번호, 정답 요지, 유형) 네 칸입니다'
        q, gold, ans, kind = [str(x).strip() for x in row]
        assert '____' not in q + gold + ans, f'{name} {i}번에 빈칸 ____ 이 남았습니다'
        assert kind in ('식별자', '바꿔말하기', '판본', '여러문서', '없음', '자유'), f'{name} {i}번 유형을 확인합니다: {kind}'
        g = [] if gold in ('없음', 'NONE', '') else [x.strip() for x in gold.split(',')]
        miss = [x for x in g if x not in IDS]
        assert not miss, f'{name} {i}번 정답 문서 번호가 재료에 없습니다: {miss}'
        assert (kind == '없음') == (not g), f'{name} {i}번: 유형이 「없음」이면 정답 문서 번호도 「없음」입니다'
        out.append(dict(id=f'{pre}{i:02d}', q=q, gold=g, answer=ans, kind=kind))
    cnt = pd.Series([o['kind'] for o in out]).value_counts().to_dict()
    print(f'✓ {name} {len(out)}문항 · {cnt}')
    return out

ev = check_set(EVALSET, '평가셋', 'E')
ts = check_set(TESTSET, '테스트셋', 'T')
assert not ({e['q'] for e in ev} & {t['q'] for t in ts}), '평가셋과 테스트셋에 같은 질문이 있습니다'
json.dump(ev, open('my_evalset.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(ts, open('my_testset.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

✓ 평가셋 20문항 · {'바꿔말하기': 5, '식별자': 4, '판본': 3, '여러문서': 3, '없음': 3, '자유': 2}
✓ 테스트셋 10문항 · {'바꿔말하기': 3, '식별자': 2, '여러문서': 2, '없음': 2, '판본': 1}


## RAG 만들기 안내

RAG 네 칸은 명세만 있고 코드는 비어 있습니다. 3 · 4일차 노트북과 강의안 예제를 보며 직접 짭니다. Claude Code 에 물어봐도 됩니다. 칸마다 아래 확인 셀이 ✓ 를 찍으면 다음 칸으로 갑니다. 확인 셀이 멈추면 오류 문구가 무엇을 고칠지 알려 줍니다.

## 쓰는 모델과 라이브러리

모델은 아래로 정해 두었고, 불러오는 코드는 완성 코드(`get_emb()` · `get_reranker()`)로 줍니다. 수강생 코드는 모델을 직접 불러오지 않고 이 두 함수를 부릅니다. 같은 모델을 두 번 불러오지 않으므로 셀을 여러 번 실행해도 됩니다.

- **임베딩(칸 B · C)** `intfloat/multilingual-e5-small` · 384차원. 문서에는 `'passage: '`, 질문에는 `'query: '` 를 앞에 붙이고, `normalize_embeddings=True` 로 정규화합니다. 3 · 4일차와 같은 모델입니다.
- **키워드 검색(칸 B · C)** `rank_bm25.BM25Okapi` · 토큰은 완성 코드의 `tok()` 을 씁니다(4일차와 같음).
- **리랭커(칸 C)** `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` · 다국어 교차 인코더, 입력 최대 512토큰. `get_reranker().predict([(질문, 청크 본문), …])` 가 쌍마다 점수 하나를 돌려줍니다. 4일차와 같은 모델입니다.
- **답변 생성(칸 D)** `llm(프롬프트, model=CONFIG['ANS_MODEL'])` · 기본 `claude-sonnet-5-5`(effort low). 「0-2」 셀의 `llm()` 이 캐시와 재시도를 맡습니다.
- **채점(평가 도구)** `claude-sonnet-5-5` 로 고정합니다. 바꾸지 않습니다.

**2단계에서 바꿔 볼 수 있는 모델** 임베딩 `intfloat/multilingual-e5-base`(768차원, 접두어 규칙 같음, 느리므로 [런타임] → [런타임 유형 변경] 에서 GPU 권장) · 리랭커 `BAAI/bge-reranker-v2-m3`(정확하지만 무거움, GPU 필요) · 답변 `claude-haiku-4-5`(빠르고 쌈). `CONFIG` 값만 바꾸고 `build_index()` 를 다시 부르면 됩니다.

**설정 블록 `CONFIG`** 2단계에서 바꿀 값을 한곳에 모읍니다. 칸 A~D 의 함수는 숫자 · 모델 이름을 코드에 직접 쓰지 말고 `CONFIG` 에서 읽습니다. 새 방식을 만들면 키를 더해도 됩니다. 평가 도구가 결과 표에 `CONFIG` 를 함께 남깁니다.

## 3. RAG 만들기 · 설정 블록과 모델 (완성 코드)

In [8]:
CONFIG = {
    'CHUNK_SIZE':   500,        # 청크 글자 수
    'OVERLAP':      0,          # 청크끼리 겹치는 글자 수
    'SEARCH':       'vector',   # ask() 가 쓰는 검색: 'vector' · 'bm25' · 'hybrid' · 'rerank'
    'TOP_K':        4,          # 답변에 넣는 발췌 수
    'CANDIDATES':   30,         # 하이브리드 · 리랭커가 각 목록에서 가져오는 후보 수
    'RRF_K':        60,         # RRF 상수
    'EMB_MODEL':    'intfloat/multilingual-e5-small',
    'RERANK_MODEL': 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
    'ANS_MODEL':    'claude-sonnet-5-5',
}

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
_MODELS = {}

def get_emb():
    """CONFIG['EMB_MODEL'] 임베딩 모델. 처음 한 번만 내려받습니다."""
    name = CONFIG['EMB_MODEL']
    if name not in _MODELS:
        _MODELS[name] = SentenceTransformer(name)
    return _MODELS[name]

def get_reranker():
    """CONFIG['RERANK_MODEL'] 리랭커. 처음 한 번만 내려받습니다."""
    name = CONFIG['RERANK_MODEL']
    if name not in _MODELS:
        _MODELS[name] = CrossEncoder(name, max_length=512)
    return _MODELS[name]

def tok(s):                     # BM25 토큰: 낱말 + 한글 두 글자씩 (4일차와 같음)
    s = re.sub(r'[^\w가-힣]+', ' ', s.lower())
    out = []
    for w in s.split():
        out.append(w)
        if len(w) > 2 and re.match(r'^[가-힣]+$', w):
            out += [w[i:i + 2] for i in range(len(w) - 1)]
    return out

print('임베딩', get_emb().get_sentence_embedding_dimension(), '차원 · 리랭커 준비', type(get_reranker()).__name__)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (14 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

임베딩 384 차원 · 리랭커 준비 CrossEncoder


## RAG 명세

평가 도구가 부르는 함수는 아래 여섯 개입니다. 이름 · 입력 · 출력이 명세와 같으면, 안을 어떻게 짰는지는 상관없습니다.

**칸 A · 청킹 `chunk_docs()`**
- 입력: 없음. 전역 `DOCS`(1. 재료 읽기에서 만든 문서 100건)를 읽습니다.
- 출력: `Chunk` 목록. `Chunk(doc_id, version, date, title, section, text)` 는 「0-2」 셀에 정의돼 있습니다. `doc_id` · `version` · `date` · `title` 은 그 청크가 나온 문서의 값을 그대로 넣고, `section` 은 청크가 속한 절 제목(모르면 빈 문자열), `text` 는 청크 본문입니다.
- 규칙: 처음에는 `CONFIG['CHUNK_SIZE']` 글자씩, `CONFIG['OVERLAP']` 글자씩 겹쳐 자릅니다(`OVERLAP` 은 `CHUNK_SIZE` 보다 작아야 합니다). 공백만 남은 청크는 버립니다. 모든 문서가 한 개 이상의 청크로 들어가야 합니다.

**칸 B · 색인 `build_index()`**
- 입력 · 출력: 없음. 전역 변수 세 개를 새로 만듭니다.
  - `chunks` · `chunk_docs()` 의 결과
  - `E` · 청크 임베딩 행렬(청크 수 × 차원). `get_emb().encode(['passage: ' + 청크 본문 …], normalize_embeddings=True)`
  - `bm25` · `BM25Okapi([tok(청크 본문) …])`
- 규칙: 끝에 청크 수와 `E.shape` 를 찍습니다. 2단계에서 청킹 · 임베딩 설정을 바꾸면 이 함수만 다시 부릅니다. `chunks` · `E` · `bm25` 는 언제나 같은 순서(같은 청크 번호)여야 합니다.

**칸 C · 검색 네 가지** 모두 입력은 질문 `q` 와 개수 `k`, 출력은 관련도가 높은 순서의 `Chunk` 목록(길이 `k`)입니다.
- `search_vector(q, k)` · 질문을 `'query: ' + q` 로 임베딩해 `E @ 질문벡터` 점수 순으로 고릅니다.
- `search_bm25(q, k)` · `bm25.get_scores(tok(q))` 점수 순으로 고릅니다.
- `search_hybrid(q, k)` · 벡터 순위와 BM25 순위를 각각 상위 `CONFIG['CANDIDATES']` 개 뽑아 RRF 로 합칩니다. 청크 하나의 점수는 「그 청크가 나온 목록마다 1 / (`CONFIG['RRF_K']` + 순위)」의 합이고, 순위는 1부터 셉니다.
- `search_rerank(q, k)` · `search_hybrid(q, CONFIG['CANDIDATES'])` 의 후보를 `get_reranker().predict([(q, 청크 본문) …])` 점수 순으로 다시 세워 상위 `k` 개를 고릅니다.
- 규칙: 동점 순서가 실행마다 바뀌지 않도록 `np.argsort(-점수, kind='stable')` 를 씁니다. `k` 가 청크 수보다 크면 있는 만큼 돌려줍니다. 평가 도구는 `k=30` 으로도 부릅니다.
- 네 함수를 다 만들면 완성 코드의 `search(q, k)` 가 `CONFIG['SEARCH']` 에 따라 넷 중 하나를 부릅니다.

**칸 D · 프롬프트와 답변 `ask(q)`**
- 입력: 질문 `q`. 출력: `(답 문자열, 발췌 Chunk 목록)` 두 개를 묶은 튜플.
- 순서: `hits = search(q, CONFIG['TOP_K'])` → 발췌 묶음을 만든다 → 프롬프트에 발췌와 질문을 넣는다 → `llm(프롬프트, model=CONFIG['ANS_MODEL'])` → `(답, hits)` 를 돌려준다. 돌려주는 `hits` 는 프롬프트에 넣은 청크 그대로입니다. 평가 도구가 이 발췌를 채점 모델에게 함께 보여 줍니다.
- 발췌 묶음: 청크마다 `[문서번호 판] (시행 날짜)` 한 줄을 앞에 붙이고 빈 줄로 구분합니다. 예: `[ET-ALM-102 v1] (시행 2025-01-21)`
- 프롬프트 규칙 넷: ① 발췌만 근거로 답한다 ② 문장마다 근거 [문서번호] 를 붙인다 ③ 같은 문서번호의 판이 둘 이상이면 시행일이 가장 늦은 판을 따른다 ④ 발췌에 답이 없으면 `REFUSE`(「문서에서 근거를 찾지 못했습니다」)를 넣은 한 문장만 답한다.
- 프롬프트 문자열에 `{` `}` 가 들어가면 `.format()` 이 오류를 냅니다. 발췌와 질문은 f-문자열이나 `.replace()` 로 넣거나, 프롬프트 안의 중괄호를 `{{ }}` 로 씁니다.

### 칸 A · 청킹 `chunk_docs()` (작성)

명세는 위 「RAG 명세」를 따릅니다. 참고: 3일차 예제 2-3(`chunk_fixed`) · 2-5(꼬리표 붙이기).

3일차의 `chunk_fixed(text, size, overlap)` 를 그대로 가져와 `DOCS` 를 도는 반복문 안에서 부르면 됩니다. 문서 한 건에서 청크를 만드는 모양은 아래와 같습니다.

```python
for d in DOCS:                      # 문서 100건 (3일차의 for nm, t in zip(names, docs))
    본문 = d['text']                 # 자를 본문
    # 청크마다 → Chunk(d['doc_id'], d['version'], d['date'], d['title'], 절 제목, 청크 본문)
```
`section`(절 제목)은 청크가 들어 있는 `## ` 줄의 제목입니다. 처음에는 빈 문자열 `''` 로 두고 시작해도 확인 셀을 통과합니다.

In [ ]:
# ✏️ 칸 A · 청킹 (작성)
def chunk_fixed(text, size=300, overlap=0):
    step = size - overlap
    return [text[i:i + size]
            for i in range(0, len(text), step)
            if text[i:i + size].strip()]

def chunk_docs():
    size, overlap = CONFIG['CHUNK_SIZE'], CONFIG['OVERLAP']
    assert 0 <= overlap < size, "OVERLAP 은 CHUNK_SIZE 보다 작아야 합니다"
    chunks = []
    for d in DOCS:                                   # 문서 100건
        본문 = d['text']
        # 본문에서 「## 절 제목」 줄의 위치를 모읍니다 → (글자 위치, 제목)
        heads = [(m.start(), m.group(1).strip()) for m in re.finditer(r'^##\s+(.+?)\s*$', 본문, re.M)]
        for i in range(0, len(본문), size - overlap):
            청크 = 본문[i:i + size]
            if not 청크.strip():                      # 공백만 남은 청크는 버립니다
                continue
            # 청크 시작 위치 이전의 마지막 ## 줄이 이 청크가 속한 절입니다(없으면 '')
            절 = ''
            for pos, title in heads:
                if pos <= i:
                    절 = title
                else:
                    break
            chunks.append(Chunk(d['doc_id'], d['version'], d['date'], d['title'], 절, 청크))
    return chunks


In [ ]:
# 칸 A 확인 (완성 코드 · 고치지 않습니다)
_c = chunk_docs()
assert isinstance(_c, list) and _c, 'chunk_docs() 는 Chunk 목록을 돌려줍니다'
assert all(isinstance(c, Chunk) for c in _c), '목록의 원소는 모두 Chunk(doc_id, version, date, title, section, text) 입니다'
assert {c.doc_id for c in _c} == {d['doc_id'] for d in DOCS}, '빠진 문서가 있습니다. 모든 문서가 청크로 들어가야 합니다'
assert all(isinstance(c.text, str) and c.text.strip() for c in _c), '빈 청크가 있습니다'
_v = {(d['doc_id'], d['version']) for d in DOCS}
assert all((c.doc_id, c.version) in _v for c in _c), 'Chunk 의 doc_id · version 은 DOCS 의 값을 그대로 씁니다'
print('✓ 청크', len(_c), '개 · 평균', int(np.mean([len(c.text) for c in _c])), '자')
print(_c[0])

### 칸 B · 색인 `build_index()` (작성)

참고: 3일차 예제 1-1 · 4일차 예제 1-3(BM25).

In [ ]:
# ✏️ 칸 B · 색인 (작성)
def build_index():
    global chunks, E, bm25
    chunks = chunk_docs()
    # 임베딩: 문서에는 'passage: ' 를 붙이고 정규화합니다 (칸 C 의 E @ 질문벡터 가 코사인 점수가 됩니다)
    E = get_emb().encode(['passage: ' + c.text for c in chunks],
                         normalize_embeddings=True, show_progress_bar=False)
    # BM25: chunks 와 같은 순서로 토큰 목록을 만듭니다
    bm25 = BM25Okapi([tok(c.text) for c in chunks])
    print('청크', len(chunks), '개 · E', E.shape)


In [ ]:
# 칸 B 확인 (완성 코드 · 고치지 않습니다)
build_index()
assert isinstance(chunks, list) and len(chunks) == E.shape[0], 'E 의 행 수는 chunks 수와 같아야 합니다'
assert abs(float(np.linalg.norm(E[0])) - 1) < 1e-3, '임베딩을 정규화합니다 (normalize_embeddings=True)'
assert len(bm25.get_scores(tok('점검'))) == len(chunks), 'bm25 는 chunks 와 같은 순서 · 같은 개수로 만듭니다'
print('✓ 색인', E.shape, '· BM25', len(chunks), '청크')
INDEX_CFG = {k: CONFIG[k] for k in ('CHUNK_SIZE', 'OVERLAP', 'EMB_MODEL')}   # 이 색인을 만든 청킹 값(평가 도구가 바뀌었는지 봅니다)

### 칸 C · 검색 네 가지 (작성)

`search_vector` · `search_bm25` · `search_hybrid` · `search_rerank` 를 만듭니다. 참고: 4일차 예제 1-2 · 1-3.

In [ ]:
# ✏️ 칸 C · 검색 (작성)
def _vec_scores(q):
    qv = get_emb().encode(['query: ' + q], normalize_embeddings=True, show_progress_bar=False)[0]
    return E @ qv                                       # 정규화돼 있으므로 코사인 점수

def _bm25_scores(q):
    return np.asarray(bm25.get_scores(tok(q)))

def _order(scores):
    return np.argsort(-np.asarray(scores), kind='stable')   # 동점이면 청크 번호가 빠른 쪽이 앞

def search_vector(q, k):
    return [chunks[i] for i in _order(_vec_scores(q))[:k]]

def search_bm25(q, k):
    return [chunks[i] for i in _order(_bm25_scores(q))[:k]]

def search_hybrid(q, k):
    n = CONFIG['CANDIDATES']
    rrf = np.zeros(len(chunks))
    for scores in (_vec_scores(q), _bm25_scores(q)):
        for rank, i in enumerate(_order(scores)[:n], 1):   # 순위는 1부터
            rrf[i] += 1 / (CONFIG['RRF_K'] + rank)
    return [chunks[i] for i in _order(rrf)[:k]]

def search_rerank(q, k):
    cands = search_hybrid(q, CONFIG['CANDIDATES'])
    scores = get_reranker().predict([(q, c.text) for c in cands])
    return [cands[i] for i in _order(scores)[:k]]


In [ ]:
# 칸 C 확인 (완성 코드 · 고치지 않습니다)
SEARCHERS = {'vector': search_vector, 'bm25': search_bm25, 'hybrid': search_hybrid, 'rerank': search_rerank}

def search(q, k=None):
    """CONFIG['SEARCH'] 에 맞는 검색 함수를 부릅니다 (완성 코드)."""
    return SEARCHERS[CONFIG['SEARCH']](q, k or CONFIG['TOP_K'])

_q = ev[0]['q']
for _name, _fn in SEARCHERS.items():
    _h = _fn(_q, 5)
    assert isinstance(_h, list) and len(_h) == 5, f'{_name}: {_fn.__name__}(q, 5) 는 Chunk 5개 목록을 돌려줍니다 (지금 {type(_h).__name__} {len(_h) if hasattr(_h, "__len__") else ""})'
    assert all(isinstance(c, Chunk) for c in _h), f'{_name}: 목록의 원소는 chunks 안의 Chunk 입니다 (번호가 아니라 Chunk)'
    assert [c.text for c in _fn(_q, 5)] == [c.text for c in _h], f'{_name}: 같은 질문에 순위가 바뀝니다 (stable 정렬)'
    assert len(_fn(_q, 30)) == min(30, len(chunks)), f'{_name}: k=30 으로 불러도 30개를 돌려줘야 합니다'
    print(f'✓ {_name:7}', [c.doc_id for c in _h])
_hy = {c.text for c in search_hybrid(_q, CONFIG['CANDIDATES'])}
assert {c.text for c in search_rerank(_q, 10)} <= _hy, 'rerank: 후보는 search_hybrid(q, CANDIDATES) 에서만 고릅니다'
print('정답 문서', ev[0]['gold'])

### 칸 D · 프롬프트와 답변 `ask(q)` (작성)

참고: 3일차 예제 3-1 · 3-2.

In [ ]:
# ✏️ 칸 D · 프롬프트와 답변 (작성)
# 발췌 묶음: 청크마다 「[문서번호 판] (시행 날짜)」 한 줄을 앞에 붙이고 빈 줄로 구분합니다 (3일차 format_ctx)
def format_ctx(hits):
    return '\n\n'.join(f'[{h.doc_id} {h.version}] (시행 {h.date})\n{h.text}' for h in hits)

# 프롬프트 틀: 역할 · 발췌 · 규칙 넷 · 질문 (3일차 TEMPLATE 에 판 규칙을 더했습니다)
TEMPLATE = """너는 반도체 공장 설비지원센터 담당자다.
아래 사내 문서 발췌만 근거로 답한다.

[발췌]
{ctx}

규칙:
1. 발췌만 근거로 답한다. 발췌에 없는 내용은 쓰지 않는다.
2. 문장마다 근거 [문서번호]를 붙인다.
3. 같은 문서번호의 판이 둘 이상이면 시행일이 가장 늦은 판을 따른다.
4. 발췌에 답이 없으면 '{refuse}' 라고만 한 문장으로 답한다.

질문: {q}"""

def ask(q):
    hits = search(q, CONFIG['TOP_K'])
    prompt = TEMPLATE.format(ctx=format_ctx(hits), q=q, refuse=REFUSE)
    return llm(prompt, model=CONFIG['ANS_MODEL']), hits


In [ ]:
# 칸 D 확인 (완성 코드 · 고치지 않습니다)
_a, _h = ask(ev[0]['q'])
assert isinstance(_a, str) and _a.strip(), 'ask(q) 의 첫째 값은 비어 있지 않은 답 문자열입니다'
assert isinstance(_h, list) and all(isinstance(c, Chunk) for c in _h), 'ask(q) 의 둘째 값은 프롬프트에 넣은 Chunk 목록입니다'
assert len(_h) == CONFIG['TOP_K'], f"발췌는 CONFIG['TOP_K'] = {CONFIG['TOP_K']}개입니다 (지금 {len(_h)}개)"
assert [c.text for c in _h] == [c.text for c in search(ev[0]['q'], CONFIG['TOP_K'])], "발췌는 search(q, CONFIG['TOP_K']) 결과 그대로입니다"
assert '[잘림]' not in _a, '답이 max_tokens 에서 잘렸습니다. llm(…, max_tokens=) 을 늘립니다'
if REFUSE not in _a:
    assert re.search(r'\[[A-Za-z]{1,4}-[A-Za-z0-9]{1,6}-\d{3}', _a), '답에 근거 [문서번호] 가 없습니다. 프롬프트 규칙 ②를 확인합니다'
_r, _ = ask('사내 구내식당의 이번 주 금요일 점심 메뉴는 무엇인가요?')
assert REFUSE in _r, f'문서에 없는 질문의 답에 REFUSE 문구가 없습니다. 프롬프트 규칙 ④를 확인합니다 → {_r[:80]}'
print('Q', ev[0]['q']); print('A', _a[:300]); print()
print('✓ 답변 형식 · 근거 표기 · 거절 모두 통과')

## 4. 평가 도구 (완성 코드 · 고치지 않습니다)

평가 도구는 완성 코드로 줍니다. 평가를 대충 만들면 개선이 진짜인지 알 수 없기 때문입니다. 평가셋은 2. 검사 셀에서 아래처럼 바뀌어 평가 도구로 들어갑니다.

- `('…', 'ET-ALM-102', '반사 전력 7% 초과 2초', '식별자')` → `{'id': 'E01', 'q': '…', 'gold': ['ET-ALM-102'], 'answer': '반사 전력 7% 초과 2초', 'kind': '식별자'}`
- 정답 문서 번호 `없음` → `gold` 가 빈 목록이 되고, 그 문항은 「거절했는가」로만 채점합니다. 검색 지표 계산에서는 빠집니다.

**검색 채점** 질문마다 `search_*(q, 30)` 의 문서번호를 앞에서부터 중복 없이 모아 문서 순위를 만듭니다. 판(v1 · v2)은 보지 않습니다.
- **hit@1 · hit@3** 상위 1 · 3 문서 안에 정답 문서가 하나라도 든 문항의 비율(%)입니다. 4일차와 같은 셈법입니다.
- **MRR** 정답 문서 가운데 가장 먼저 나온 것의 순위 역수(1위 1, 2위 0.5 …)의 평균입니다.
- **recall@3** 상위 3 문서에 든 정답 문서 수 ÷ 그 문항의 정답 문서 수의 평균(%)입니다. 정답 문서가 둘인 「여러문서」 문항은 하나만 찾으면 0.5 입니다.
- `run_retrieval()` 은 LLM 없이 검색 네 가지(vector · bm25 · hybrid · rerank)를 한 표로 잽니다.

**답변 채점 · 정답률(`correct`)** `ask(q)` 의 답과 발췌, 정답 요지, 정답 문서의 최신 판 원문을 채점 모델에게 주고 「정답성: 예 · 부분 · 아니오」를 받습니다. 「예」만 정답으로 셉니다. 정답 문서가 `없음` 인 문항은 답에 `REFUSE` 문구가 있으면 「거절 성공」입니다.

**결과 칸** 문항마다 검색과 답을 함께 봐서 한 칸에 넣습니다. 「찾고」는 `ask(q)` 의 발췌에 정답 문서가 하나라도 들어 있었다는 뜻입니다.
- **찾고 맞음** 잘 된 문항
- **찾고 틀림** 발췌는 맞았는데 답이 틀림 → 프롬프트 · 발췌 수 · 청크 경계 쪽 문제
- **못 찾고 틀림** 정답 문서를 못 찾음 → 청킹 · 검색 방식 쪽 문제
- **못 찾고 맞음** 다른 문서로 우연히 맞힘 → 근거가 틀렸으니 확인 대상
- **거절 성공 · 거절 실패** 「없음」 문항
- **오류** 그 문항에서 내 코드가 예외를 냈습니다. 평가는 멈추지 않고 다음 문항으로 가며, `show_fail()` 이 오류 문구를 보여 줍니다.

**실행하면 남는 것** `results.csv` 에 한 줄(태그 · 검색 방식 · 검색 지표 · 정답률 · 거절 · 결과 칸 수 · 그때의 `CONFIG`), `fail_<태그>.csv` 에 문항별 결과.

In [ ]:
# @title 4. 평가 도구 (완성 코드)
assert 'ask' in globals(), '칸 D 셀과 칸 D 확인 셀을 먼저 실행합니다'
# 4. 평가 도구: 검색 지표(hit@k · MRR)를 계산하고, 채점 모델로 답의 정답성을 판정해 결과 칸을 나눕니다
JUDGE_MODEL = 'claude-sonnet-5-5'
# 채점 프롬프트: 정답 요지와 정답 문서 원문을 주고 「정답성: 예/아니오/부분」 한 줄만 받습니다
JUDGE = """아래는 사내 문서 기반 질의응답의 결과다.

[질문] {q}

[모델이 받은 발췌]
{ctx}

[모델 답변]
{ans}

[정답 요지와 정답 문서 원문]
{gold}

정답성: 정답 요지와 원문에 비추어 답이 사실로 맞는가 (핵심 값이 맞으면 예, 일부만 맞으면 부분)
출력 형식(설명 금지):
정답성: 예/아니오/부분"""
# 문서번호마다 가장 최근 판을 골라 둡니다(채점 때 최신 판 원문을 정답 근거로 씁니다)
LATEST = {}
for d in DOCS:
    if d['doc_id'] not in LATEST or d['date'] > LATEST[d['doc_id']]['date']:
        LATEST[d['doc_id']] = d
# 결과 칸 이름(4일차 · 실습 미션 1 과 같은 이름)
CELLS = ['찾고 맞음', '찾고 틀림', '못 찾고 틀림', '못 찾고 맞음', '거절 성공', '거절 실패', '오류']

# 청크 순위를 문서 순위로 바꿉니다(같은 문서는 한 번만 셉니다)
def _doc_rank(fn, q):
    seen = []
    for c in fn(q, 30):
        if c.doc_id not in seen:
            seen.append(c.doc_id)
    return seen

def _metrics(fn, S):
    """hit@1 · hit@3 · MRR (4일차와 같은 셈법)과 보조 지표 recall@3."""
    hit = {1: 0, 3: 0}; recall3 = 0.0; mrr = 0.0; n = 0
    for e in S:
        if not e['gold']:
            continue
        rk = _doc_rank(fn, e['q']); n += 1
        for k in hit:
            hit[k] += any(d in e['gold'] for d in rk[:k])
        recall3 += len(set(rk[:3]) & set(e['gold'])) / len(e['gold'])
        # 정답 문서가 처음 나온 순위(0부터 셈)이고, 없으면 None 입니다
        first = next((j for j, d in enumerate(rk) if d in e['gold']), None)
        mrr += 0 if first is None else 1 / (first + 1)
    return {'hit@1': round(hit[1] / n * 100, 1), 'hit@3': round(hit[3] / n * 100, 1),
            'mrr': round(mrr / n, 3), 'recall@3': round(recall3 / n * 100, 1)}

def run_retrieval(S=None):
    """검색 네 가지를 LLM 없이 비교합니다."""
    rows = [{'search': name} | _metrics(fn, S or ev) for name, fn in SEARCHERS.items()]
    T = pd.DataFrame(rows)
    print(T.to_string(index=False))
    return T

# 발췌를 「[문서번호 판] (시행일) + 본문」 꼴로 이어 붙여 채점 프롬프트에 넣습니다
def _ctx(hits):
    return '\n\n'.join(f'[{h.doc_id} {h.version}] (시행 {h.date})\n{h.text}' for h in hits)

def _answer(e):
    """수강생 코드(ask)를 부릅니다. 임베딩 · 리랭커가 겹쳐 돌면 커널이 죽을 수 있어 한 문항씩 차례로 부릅니다."""
    row = dict(id=e['id'], kind=e['kind'], q=e['q'])
    try:
        ans, hits = ask(e['q'])
        assert isinstance(ans, str), 'ask(q) 의 첫째 값이 문자열이 아닙니다'
        pos = {id(c): j for j, c in enumerate(chunks)}
        lab = [f"{h.doc_id} {h.version} · " + (h.section[:14] if h.section else f"청크 {pos.get(id(h), '?')}") for h in hits]
        row |= dict(ans=ans, hits=[h.doc_id for h in hits], hits_label=lab, _ctx=_ctx(hits))
    except Exception as ex:
        row |= dict(cell='오류', ans='', hits=[], verdict=f'{type(ex).__name__}: {ex}'[:300])
    return row

def _judge(row, e):
    """채점 모델 호출(LLM 만 부르므로 병렬로 실행합니다)."""
    if row.get('cell') == '오류':
        return row
    if not e['gold']:
        return row | dict(cell='거절 성공' if REFUSE in row['ans'] else '거절 실패', verdict='거절 문항')
    # 정답 문서마다 앞 6,000자씩 넣습니다(제공 문서는 모두 6,000자 안이라 전부 들어갑니다).
    # 이은 뒤에 자르면 「여러문서」 문항의 둘째 문서가 잘려 채점 근거가 빠집니다
    gold = f"[정답 요지] {e['answer']}\n\n" + '\n\n'.join(LATEST[g]['text'][:6000] for g in e['gold'])
    try:
        v = llm(JUDGE.format(q=e['q'], ctx=row['_ctx'][:12000], ans=row['ans'], gold=gold), model=JUDGE_MODEL, max_tokens=30)
    except Exception as ex:
        return row | dict(cell='오류', verdict=f'채점 호출 실패 {type(ex).__name__}: {ex}'[:300])
    # 발췌에 정답 문서가 있었는가(찾음) × 채점이 「예」인가(맞음)로 결과 칸을 정합니다
    found = any(g in row['hits'] for g in e['gold'])
    ok = bool(re.search(r'정답성:\s*예', v))
    return row | dict(cell=('찾고 ' if found else '못 찾고 ') + ('맞음' if ok else '틀림'), verdict=v.strip())

# 한 세트(평가셋 또는 테스트셋)를 끝까지 실행해 results.csv 에 한 줄, fail_<태그>.csv 에 문항별 결과를 남깁니다
def _run(S, tag, which):
    t0 = time.time()
    rec = _metrics(search, S)
    answered = [_answer(e) for e in S]                       # 수강생 코드는 차례로
    with ThreadPoolExecutor(4) as ex:                         # 채점 LLM 만 병렬로
        rows = list(ex.map(_judge, answered, S))
    for r in rows:
        r.pop('_ctx', None)
        r['ans'] = r['ans'][:400]
    # 칸별 문항 수와 정답률(답이 있는 문항 가운데 맞음 비율)을 계산합니다
    cnt = {c: sum(r['cell'] == c for r in rows) for c in CELLS}
    n_have = sum(1 for e in S if e['gold'])
    res = dict(set=which, tag=tag, search=CONFIG['SEARCH'], **rec,
               correct=round(sum(r['cell'] in ('찾고 맞음', '못 찾고 맞음') for r in rows) / n_have * 100, 1),
               refuse_ok=f"{cnt['거절 성공']}/{sum(1 for e in S if not e['gold'])}",
               refuse_have=f"{sum(1 for r, e in zip(rows, S) if e['gold'] and REFUSE in r['ans'])}/{n_have}",
               **{c: cnt[c] for c in CELLS}, chunks=len(chunks),
               config=json.dumps(CONFIG, ensure_ascii=False), sec=round(time.time() - t0), at=time.strftime('%Y-%m-%d %H:%M'))
    # results.csv 에 이번 결과를 한 줄로 더합니다
    table = pd.read_csv('results.csv', encoding='utf-8-sig').to_dict('records') if os.path.exists('results.csv') else []
    same = [i for i, r in enumerate(table) if r.get('set') == which and r.get('tag') == tag]
    if same:                                                  # 같은 태그를 다시 측정하면 그 줄을 덮어씁니다(두 번 쌓이지 않음)
        table[same[0]] = res
        table = [r for i, r in enumerate(table) if i not in same[1:]]
    else:
        table.append(res)
    pd.DataFrame(table).to_csv('results.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(rows).to_csv(f'fail_{tag}.csv', index=False, encoding='utf-8-sig')
    print({k: res[k] for k in ('set', 'tag', 'search', 'hit@1', 'hit@3', 'mrr', 'recall@3', 'correct', 'refuse_ok', 'refuse_have')})
    print(' · '.join(f'{c} {cnt[c]}' for c in CELLS if cnt[c] or c != '오류'))
    if cnt['오류']:
        print(f"⚠ 오류 {cnt['오류']}건: 아래 문항별 줄에서 「오류」 문항의 채점 뒤 문구를 보고 고친 셀을 다시 실행합니다")
    return rows

def run_eval(tag):
    """평가셋 · 지금 CONFIG['SEARCH'] 의 검색으로 답까지 채점합니다. 2단계에서 몇 번이든 실행합니다."""
    return _run(ev, tag, 'eval')

def show_fail(rows, only=None):
    """맞지 않은 문항을 짧게 보여 줍니다. only 에 문항 번호 목록을 주면 그 문항만 보여 줍니다."""
    OKC = ('찾고 맞음', '못 찾고 맞음', '거절 성공')
    pick = [r for r in rows if (r['id'] in only if only is not None else r['cell'] not in OKC)]
    for r in pick:
        q = r['q'] if len(r['q']) <= 46 else r['q'][:45] + '…'
        ans = ' '.join(str(r.get('ans', '')).split())
        lab = r.get('hits_label') or r.get('hits') or []
        if isinstance(lab, str):
            lab = [lab]
        print(f"{r['id']} [{r['kind']}] {r['cell']} · {q}")
        print(f"   발췌 {' | '.join(lab)}")
        print(f"   답 {ans[:110]}{'…' if len(ans) > 110 else ''} · 채점 {str(r.get('verdict', '')).replace('정답성: ', '')}")
    bad = [r['id'] for r in rows if r['cell'] not in OKC]
    print(f"맞지 않은 문항 {len(bad)} / {len(rows)}: {' '.join(bad)}")

# results.csv 에 쌓인 줄을 한 표로 보여 줍니다
def show_results():
    R = pd.read_csv('results.csv', encoding='utf-8-sig')
    cols = ['set', 'tag', 'search', 'hit@1', 'hit@3', 'mrr', 'recall@3', 'correct', 'refuse_ok', 'refuse_have'] + CELLS[:4]
    print(R[cols].to_string(index=False))
# 청킹 · 임베딩 값(CHUNK_SIZE · OVERLAP · EMB_MODEL)을 바꾼 뒤 build_index() 를 부르지 않았으면 측정 전에 색인을 다시 만듭니다
_run0 = _run
def _run(S, tag, which):
    global INDEX_CFG
    if any(INDEX_CFG.get(k) != CONFIG.get(k) for k in INDEX_CFG):
        print('청킹 · 임베딩 값이 바뀌어 색인을 다시 만듭니다'); build_index()
        INDEX_CFG = {k: CONFIG[k] for k in INDEX_CFG}
    return _run0(S, tag, which)
print('평가 도구 준비 · 평가셋', len(ev), '문항')

## 5. 검색기 채점 출력 (완성 코드)

먼저 검색 네 가지를 LLM 없이 비교하고, 이어서 `CONFIG['SEARCH']` 로 정한 검색(처음에는 vector)으로 답까지 채점합니다. 2단계의 모든 개선은 이 두 결과와 비교합니다.

In [ ]:
# @title 5. 검색기 채점 출력 · 검색 네 가지 (완성 코드)
RETRIEVAL_TABLE = run_retrieval()

In [ ]:
# @title 5. 검색기 채점 출력 · 답변과 채점 (완성 코드)
base = run_eval('검색기채점')
show_fail(base)

# 2단계 · 평가하고 고치기

개선을 **세 번 이상** 합니다. 한 번에 하나만 바꾸고, 바꿀 때마다 같은 평가셋으로 다시 잽니다.

**개선 한 번의 순서**
1. **현재 문제** `show_fail()` 에서 어느 칸 · 어느 문항이 틀렸는지, 발췌와 답을 읽고 원인을 적습니다. 「못 찾고 틀림」이 많으면 청킹 · 검색 쪽, 「찾고 틀림」이 많으면 프롬프트 · 발췌 쪽부터 봅니다.
2. **고칠 계획** 무엇을 왜 바꾸는지, 어느 문항이 나아질지 예상을 적습니다.
3. **바꾼 점** 칸 A~D 의 코드를 고치거나 `CONFIG` 값을 바꿉니다. 평가셋 문구를 고쳤다면 그 이유도 적습니다.
4. **결과** `run_retrieval()` 의 검색 지표와 `run_eval()` 의 정답률 · 결과 칸을 전 → 후로 적고, 맞게 바뀐 문항 · 새로 틀린 문항을 적은 뒤 남길지 되돌릴지 정합니다. 점수가 떨어졌으면 되돌리고, 그것도 결과로 적습니다.

**고칠 수 있는 곳(예시)** 검색 방식(`CONFIG['SEARCH']` 를 hybrid · rerank 로) · 하이브리드 후보 수 · RRF 상수 · 청크 크기 · 겹침 · 절 단위 청킹 · 청크 앞에 문서 제목 붙이기 · 발췌 수 · 프롬프트 규칙 · 임베딩 · 리랭커 · 답변 모델 · 평가셋 문구. 이 밖의 방법도 됩니다.

**고친 뒤 다시 잴 때** 칸 A · B 를 고쳤으면 `build_index()` 를 다시 부릅니다. 함수를 새로 정의한 셀은 실행해야 반영됩니다. 칸 C 를 고쳤으면 「칸 C 확인」 셀을 다시 실행해 `SEARCHERS` 를 새 함수로 바꿉니다.

## 6. 개선 (작성 · 세 번 이상)

개선마다 셀을 하나씩 씁니다. 셀 맨 위에 무엇을 바꾸는지 주석을 달고, 마지막 줄에서 평가를 돌립니다.

In [ ]:
# ✏️ 첫 번째 개선 · 바꾸는 것: CONFIG['SEARCH'] = 'hybrid'
# (코드 수정 또는 CONFIG 변경)
CONFIG['SEARCH'] = 'hybrid'
run_retrieval()
imp1 = run_eval('개선1')
show_fail(imp1)

In [ ]:
# ✏️ 두 번째 개선 · 바꾸는 것:CONFIG['SEARCH'] = 'rerank'
# (코드 수정 또는 CONFIG 변경)
CONFIG['SEARCH'] = 'rerank'
run_retrieval()
imp2 = run_eval('개선2')
show_fail(imp2)

In [ ]:
# ✏️ 세 번째 개선 · 바꾸는 것: 청킹을 「## 절 단위」로 바꾸고, 청크마다 문서 제목 · 절 제목을 붙이고, 긴 절만 겹쳐 자른다
# (코드 수정 + CONFIG 변경)
CONFIG['SEARCH'] = 'rerank'
CONFIG['CHUNK_SIZE'] = 800          # 절이 이보다 길 때만 다시 자릅니다
CONFIG['OVERLAP'] = 100

def chunk_docs():
    size, overlap = CONFIG['CHUNK_SIZE'], CONFIG['OVERLAP']
    assert 0 <= overlap < size, "OVERLAP 은 CHUNK_SIZE 보다 작아야 합니다"
    chunks = []
    for d in DOCS:
        # 본문을 「## 절」 단위로 나눕니다. 첫 조각은 첫 ## 앞의 머리말일 수 있습니다
        for piece in re.split(r'\n(?=## )', d['text']):
            piece = piece.strip()
            if not piece:
                continue
            m = re.match(r'^##\s+(.+?)\s*(?:\n|$)', piece)
            절 = m.group(1) if m else ''
            body = piece[m.end():] if m else piece
            head = f"{d['title']} › {절}" if 절 else d['title']      # 모든 청크 앞에 붙는 문맥 줄
            # 절이 짧으면 통째로 한 청크, 길면 겹쳐서 자릅니다
            parts = [body] if len(body) <= size else chunk_fixed(body, size, overlap)
            for p in parts:
                if p.strip():
                    chunks.append(Chunk(d['doc_id'], d['version'], d['date'], d['title'], 절, head + '\n' + p.strip()))
    return chunks

build_index()
INDEX_CFG = {k: CONFIG[k] for k in INDEX_CFG}      # 평가 도구가 색인을 한 번 더 만들지 않게 합니다
run_retrieval()
imp3 = run_eval('개선3')
show_fail(imp3)


In [ ]:
# 지금까지 쌓인 결과를 한 표로 봅니다
show_results()

## 7. 테스트 (완성 코드 · 한 번만)

테스트용은 개선과정에선 안쓰고, 마지막에 한 번만 사용합니다. 개선을 마치고 최종 설정을 정한 뒤 실행합니다.

In [ ]:
# @title 7. 테스트 (완성 코드)
final = run_eval('최종설정_평가셋')     # 최종 설정의 평가셋 점수를 한 번 더 남겨 테스트와 나란히 봅니다
test = _run(ts, '테스트', 'test')
print('\n테스트셋 검색 네 가지'); run_retrieval(ts)
show_fail(test)
show_results()

## 8. ✏️ 개선내역 (작성)

이 셀을 두 번 눌러 열고 적습니다. 숫자는 `run_retrieval()` · `show_results()` 표에서, 문항 번호는 `show_fail()` 에서 옮깁니다. 개선이 네 번 이상이면 같은 틀을 이어 붙입니다. 각 칸에 쓰는 것은 바로 아래 셀에 있습니다.

### 현재 결과:

### 첫 번째 개선
- 현재 문제: 검색기채점(vector) 결과는 정답률 29.4%, 찾고 맞음 4 · 찾고 틀림 4 · 못 찾고 틀림 8 이다. 못 찾고 틀림이 가장 많아 청킹 · 검색 쪽 문제다. 검색 네 가지 표에서 키워드 검색(bm25)의 hit@1 이 47.1% 로 임베딩(vector) 23.5% 의 두 배라, 임베딩만으로는 알람 코드나 문서번호 같은 식별자 문항을 놓친다.
- 고칠 계획: `CONFIG['SEARCH']` 를 `'hybrid'` 로 바꿔 벡터 순위와 BM25 순위를 RRF 로 합친다. 두 신호가 서로 못 잡는 문항을 채워 hit@3 · 못 찾고 틀림이 줄 것으로 예상한다.
----
- 바꾼 점: `CONFIG['SEARCH'] = 'hybrid'` 한 줄(코드 수정 없음).
- 결과: hit@3 41.2 → 70.6, MRR 0.368 → 0.514, 정답률 29.4 → 47.1. 결과 칸은 찾고 맞음 4 → 8, 찾고 틀림 4 → 5, 못 찾고 틀림 8 → 4 로 못 찾는 문항이 절반으로 줄었다. 다만 hit@1 은 29.4% 로 bm25(47.1%)보다 낮아, 정답 문서가 상위에 들어와도 1등이 아닌 경우가 많다. 점수가 올랐으므로 남긴다.

### 두 번째 개선
- 현재 문제: hybrid 에서 hit@3 70.6% 인데 hit@1 은 29.4% 로, 정답 문서가 후보 안에는 있지만 순위가 낮다. 아직 못 찾고 틀림이 4건(E02 · E04 · E09 · E19), 찾고 틀림이 5건이다. E02 · E04 는 알람 코드로 묻는 식별자 문항인데 정답 문서가 발췌 4개에 들지 못했다.
- 고칠 계획: `CONFIG['SEARCH']` 를 `'rerank'` 로 바꿔 hybrid 후보 30개를 리랭커가 질문과 함께 읽고 다시 세우게 한다. 순위가 좋아져 E02 · E04 처럼 후보에는 있던 문항이 풀릴 것으로 예상한다.
----
- 바꾼 점: `CONFIG['SEARCH'] = 'rerank'` 한 줄(코드 수정 없음).
- 결과: hit@1 29.4 → 64.7, hit@3 70.6 → 88.2, MRR 0.514 → 0.764 로 검색 지표는 크게 올랐다. 정답률은 47.1 → 52.9 로 조금만 올랐다. 결과 칸은 찾고 맞음 8 → 9, 찾고 틀림 5 → 6, 못 찾고 틀림 4 → 2. E02 · E04 는 맞게 바뀌었고, E05 는 새로 틀렸다(정답 문서는 찾았으나 답이 거절). 검색 점수가 올랐으므로 남긴다. 검색이 좋아진 만큼 정답률이 오르지 않은 것은 문서를 찾아도 답이 틀리는 문제가 남았다는 뜻이다.

### 세 번째 개선
- 현재 문제: 찾고 틀림 6건(E05 · E06 · E10 · E13 · E14 · E15)이고, 정답이 있는 17문항 중 8문항에서 모델이 거절했다(`refuse_have` 8/17). 답을 보면 청크가 중간에서 잘려 값이 발췌에 없다. 개선1 의 E06 은 「헬륨 항목이 표에서 잘렸다」, E15 는 「마지막 빔에서 잘려 있어」 라고 답했다. 청크를 500자 고정, 겹침 0 으로 잘라 값이 경계에서 끊긴다. E05 · E06 · E10 은 정답 문서의 청크가 발췌에 있는데도 거절했다.
- 고칠 계획: 청킹을 `## 절` 단위로 바꾸고, 절이 길 때만 겹쳐 자른다. 청크마다 앞에 `문서 제목 › 절 제목` 줄을 붙여 임베딩 · BM25 가 문맥을 보게 한다. 값이 한 청크 안에 들어와 E05 · E06 · E10 · E13 이 풀리고, 정답이 있는데 거절하는 비율이 줄 것으로 예상한다.
----
- 바꾼 점: 코드 수정과 `CONFIG` 변경. `chunk_docs()` 를 절 단위 청킹으로 다시 정의했다(절이 `CHUNK_SIZE` 이하면 한 청크, 넘으면 `chunk_fixed` 로 자름). 청크마다 `문서 제목 › 절 제목` 줄을 붙였고, `CHUNK_SIZE` 500 → 800, `OVERLAP` 0 → 100 으로 바꿨다. 한 번에 하나만 바꾸라는 규칙에서 「청킹」 한 가지로 묶어 바꿨으므로, 절 단위 · 제목 줄 · 크기 · 겹침 중 어느 요소가 효과를 냈는지는 따로 재지 않았다.
- 결과: hit@1 64.7 → 82.4, hit@3 88.2 → 100.0, MRR 0.764 → 0.902, recall@3 82.4 → 94.1, 정답률 52.9 → 88.2. 결과 칸은 찾고 맞음 9 → 15, 찾고 틀림 6 → 2, 못 찾고 틀림 2 → 0. 정답이 있는데 거절한 문항은 8/17 → 2/17. 청크 수는 827개. 맞게 바뀐 문항은 E05 · E06 · E09 · E10 · E13 · E19, 새로 틀린 문항은 없다. 남은 E14 · E15 는 여러문서 문항으로, 발췌 4개에 정답 문서 둘 중 하나만 들어왔다(E14 는 주간 PM 점검표 대신 월간 PM, E15 는 LOTO 문서 누락). 점수가 크게 올랐으므로 남긴다.

### 테스트 진행 (테스트용 평가셋 10문항)
- 결과: 같은 최종 설정(rerank · 절 단위 청킹)으로 테스트셋을 한 번 돌렸다. 검색 지표는 hit@1 87.5 · hit@3 87.5 · MRR 0.906 · recall@3 75.0, 정답률은 62.5%(정답이 있는 8문항 중 5문항 정답), 거절 2/2 성공이다. 결과 칸은 찾고 맞음 5 · 찾고 틀림 2 · 못 찾고 틀림 1 이다. 같은 설정의 평가셋은 정답률 88.2%, hit@3 100.0% 였으므로 정답률은 25.7%p 낮고, hit@1 · MRR 은 평가셋과 비슷하다. 틀린 문항은 T02 · T07 · T08 이다.
  - T02(식별자): 질문에 문서번호 IM-CASE-703 이 있는데 발췌에 IM-CASE-706 · 709 가 들어와 정답 문서를 못 찾았다.
  - T07 · T08(여러문서): 정답 문서 둘 중 하나만 발췌에 들어왔고 나머지 값(센터장 처리 기한, 챔버 냉각 시간)은 근거를 찾지 못했다고 답했다. 평가셋에서 마지막까지 남은 E14 · E15 와 같은 유형이다.
  - 검색 네 가지 비교에서는 이 테스트셋의 hit@3 이 vector 100.0, rerank 87.5 였다. 정답 문서가 있는 문항이 8개라 한 문항 차이이므로 방식의 우열로 읽지 않는다.

### 만든 RAG에 대한 총평
한 문서 안에 답이 있는 질문(식별자 · 바꿔말하기 · 판본)은 평가셋 · 테스트셋 모두 잘 답했고, 문서에 없는 질문은 5문항 모두 거절에 성공했다. 두 문서를 함께 봐야 하는 질문은 평가셋 2건 · 테스트셋 2건이 모두 둘째 문서가 발췌에서 빠져 틀렸다. 평가셋 88.2% 와 테스트셋 62.5% 의 차이는 평가셋을 보며 고쳐서 평가셋에 맞춰진 부분이 있고, 테스트셋이 10문항이라 한 문항이 10~12.5%p 를 움직이기 때문으로 본다(원인을 따로 실험하지는 않았다). 실제 업무에 쓰려면 여러 문서가 필요한 질문에서 문서마다 발췌를 확보하는 방법(질문을 나눠 검색하기, 발췌 수 늘리기)과, 문서번호로 직접 묻는 질문을 위해 청크에 문서번호를 넣는 개선, 더 많은 질문으로 하는 평가가 필요하다.


**각 칸에 쓰는 것**
- **현재 결과** 비워 둡니다.
- **현재 문제** 결과 칸과 문항 번호, 발췌와 답을 보고 찾은 원인. (예) 「찾고 틀림 7건. E04 · E09 는 승인 단계표가 두 청크로 잘려 2 · 3단계 결재자가 발췌에 없었다」
- **고칠 계획** 무엇을 왜 바꾸는지와 예상. (예) 「절 단위로 자르면 승인 단계표가 한 청크에 들어가 E04 · E09 가 풀릴 것」
- **바꾼 점** 바꾼 코드나 `CONFIG` 값. 한 번에 하나만.
- **결과** hit@3 · 정답률 전 → 후, 결과 칸(찾고 맞음 / 찾고 틀림 / 못 찾고 틀림) 전 → 후, 맞게 바뀐 문항 · 새로 틀린 문항, 남김 또는 되돌림.
- **테스트 진행 결과** 테스트셋 검색 지표 · 정답률 · 결과 칸 · 거절과, 같은 설정의 평가셋 점수(`최종설정_평가셋` 줄)를 나란히 적습니다.
- **총평** 내 RAG 가 잘 답하는 질문과 못 답하는 질문, 평가셋과 테스트셋 점수 차이의 이유, 실제 업무에 쓰려면 더 필요한 것을 세 문장 안팎으로 씁니다.

## 9. 제출

아래 파일을 내려받아 `제출_aaa.zip`(aaa 자리에 내 식별자) 하나로 묶어 휴넷 랩스에 올립니다.
1. 노트북: [파일] → [다운로드] → [.ipynb 다운로드] 로 받습니다. 「8. 개선내역」 셀까지 채우고 실행 출력이 남은 채로 받습니다.
2. 결과 파일: 왼쪽 📁 파일 창에서 `results.csv` 와 `fail_` 로 시작하는 파일을 모두 [다운로드] 합니다(파일 이름 옆 ⋮ → [다운로드]).
3. 내려받은 파일을 한 폴더에 모아 zip 으로 묶습니다.

# 3단계 · *내 업무로* 재료와 평가셋 만들기(선택)

제출을 마친 뒤 시간이 남으면 합니다. 채점하지 않습니다. 재료와 평가셋만 내 것으로 바꾸고, 칸 A~D 와 평가 도구는 그대로 씁니다.

**재료 파일 규칙**
- 문서 한 건 = md 파일 한 개(UTF-8). 실제 사내 문서 · 실명 · 실제 수치는 넣지 않습니다.
- 파일 맨 위는 아래 꼴로 씁니다. 「- 문서번호:」 값이 평가셋의 정답 문서 번호가 됩니다. 정보 줄이 없으면 파일 이름(`MY-GDE-001_v2.md` → 문서번호 `MY-GDE-001`, 판 `v2`)을 씁니다.
```
# MY-GDE-001 사내 챗봇 장애 대응 가이드
- 문서번호: MY-GDE-001
- 판: v1
- 시행일: 2025-03-04
- 종류: 운영 가이드

## 1. 절 제목
본문 …
```
- 개정판은 같은 문서번호에 `- 판: v2` 와 더 늦은 시행일을 씁니다.

**바꾸는 순서**
1. 제출할 파일을 모두 내려받았는지 확인하고, [런타임] → [런타임 연결 해제 및 삭제] 를 누릅니다. 파일 창의 재료와 결과 파일이 지워집니다.
2. 내 md 파일을 모두 zip 하나로 묶어(예: `내재료.zip`) 파일 창 맨 위 칸에 올립니다.
3. 「0. 재료 받기」 셀의 `DOCS_ZIP = 'day05-mini-project-docs.zip'` 를 `DOCS_ZIP = '내재료.zip'` 으로 바꿉니다. 셀이 전에 받은 재료(`mini_docs/`)를 지우고 내 zip 을 그 자리에 풉니다.
4. 「2. 평가셋 · 테스트셋」 셀의 `EVALSET` · `TESTSET` 을 내 문서로 쓴 문항으로 바꿉니다. 문항 수는 정해져 있지 않습니다.
5. 맨 위 셀부터 다시 실행합니다.